In [10]:
import os

DATA_PATH = "data"

# create folder if not exists
os.makedirs(DATA_PATH, exist_ok=True)

files = os.listdir(DATA_PATH)

if len(files) < 4:
    print("⚠️ Dataset not found. Please upload the 4 .txt files.")

    from google.colab import files as colab_files
    uploaded = colab_files.upload()

    for filename in uploaded.keys():
        with open(os.path.join(DATA_PATH, filename), "wb") as f:
            f.write(uploaded[filename])

    print("✅ Files uploaded successfully!")

else:
    print("✅ Dataset found locally:", files)

✅ Dataset found locally: ['Document_3_Stakeholder_Memo.txt', 'Document_1_Policy_Report.txt', 'Document_4_Technical_Brief.txt', 'Document_2_News_Article.txt']


In [11]:
data_path = "data"

In [12]:
%%writefile data_ingestion.py
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
import os


def clean_text(text):
    text = text.replace("\n", " ")
    text = text.replace("  ", " ")
    return text.strip()


def load_documents(data_path):
    docs = []
    for file in os.listdir(data_path):
        path = os.path.join(data_path, file)
        loader = TextLoader(path)
        loaded = loader.load()

        for doc in loaded:
            doc.page_content = clean_text(doc.page_content)
            doc.metadata["source"] = file

            # timestamp priority
            if "doc2" in file:
                doc.metadata["priority"] = 3
            elif "doc4" in file:
                doc.metadata["priority"] = 3
            else:
                doc.metadata["priority"] = 1

        docs.extend(loaded)

    return docs


def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    return splitter.split_documents(docs)


def ingest(data_path="data"):
    docs = load_documents(data_path)
    chunks = split_documents(docs)

    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(chunks, embeddings)

    return vectorstore, embeddings

Writing data_ingestion.py


In [13]:
%%writefile router.py
import numpy as np

SYNTHESIS_KEYWORDS = [
    "compare", "difference", "across", "contrast",
    "how do", "relationship", "differ"
]


def route_query(query, vectorstore, embeddings):
    results = vectorstore.similarity_search_with_score(query, k=5)

    scores = [score for _, score in results]
    avg_score = np.mean(scores)

    reasoning = {
        "avg_score": avg_score,
        "top_scores": scores
    }

    # Rule 1: Out-of-scope
    if avg_score > 0.8:
        reasoning["decision"] = "Low similarity → Out of scope"
        return "out_of_scope", reasoning

    # Rule 2: Synthesis
    if any(word in query.lower() for word in SYNTHESIS_KEYWORDS):
        reasoning["decision"] = "Keyword-based synthesis"
        return "synthesis", reasoning

    # Rule 3: Multi-chunk signal
    if len(results) >= 3:
        reasoning["decision"] = "Multiple relevant chunks → synthesis"
        return "synthesis", reasoning

    reasoning["decision"] = "Default factual"
    return "factual", reasoning

Overwriting router.py


In [14]:
%%writefile retrieval.py
def retrieve(query, vectorstore, mode):
    if mode == "factual":
        return vectorstore.similarity_search(query, k=2)

    if mode == "synthesis":
        return vectorstore.max_marginal_relevance_search(query, k=5)

    return []

Overwriting retrieval.py


In [15]:
%%writefile generator.py
from openai import OpenAI

client = OpenAI()


def generate_answer(query, docs, mode):
    if mode == "out_of_scope":
        return "The provided documents do not contain enough information to answer this question."

    # sort by priority (newer/more reliable docs first)
    docs = sorted(docs, key=lambda d: d.metadata.get("priority", 0), reverse=True)

    context = "\n\n".join([
        f"[Source: {d.metadata.get('source')}]\n{d.page_content}"
        for d in docs
    ])

    prompt = f"""
You are answering strictly from provided documents.

Rules:
- If multiple sources disagree → mention both and prefer newer info
- Do NOT hallucinate
- Cite sources

Context:
{context}

Question: {query}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

Overwriting generator.py


In [16]:
%%writefile test_questions.py
test_set = [

# FACTUAL
{"query": "What are the four risk categories in the EU AI Act?", "type": "factual", "expected": "unacceptable high limited minimal"},
{"query": "What is the penalty for prohibited AI systems under the EU AI Act?", "type": "factual", "expected": "35 million euros 7%"},
{"query": "What defines a frontier AI model?", "type": "factual", "expected": "10^26"},
{"query": "What AI systems are banned?", "type": "factual", "expected": "social scoring biometric"},
{"query": "What must training data avoid in China?", "type": "factual", "expected": "copyright violation"},

# SYNTHESIS
{"query": "Compare EU and US AI regulation approaches", "type": "synthesis", "expected": "EU strict US sectoral"},
{"query": "What are common themes in AI regulation?", "type": "synthesis", "expected": "transparency accountability"},
{"query": "How do different documents describe high-risk AI?", "type": "synthesis", "expected": "defined ambiguous"},
{"query": "How do China EU and US regulate AI differently?", "type": "synthesis", "expected": "China strict EU comprehensive US flexible"},
{"query": "What is the EU AI Act timeline?", "type": "synthesis", "expected": "2024 2025 2026 2027"},

# OUT OF SCOPE
{"query": "What is quantum computing?", "type": "out_of_scope", "expected": None},
{"query": "What is India's AI policy?", "type": "out_of_scope", "expected": None},
{"query": "Explain blockchain", "type": "out_of_scope", "expected": None},
{"query": "Who invented AI?", "type": "out_of_scope", "expected": None},
{"query": "What is climate policy?", "type": "out_of_scope", "expected": None},
]

Overwriting test_questions.py


In [17]:
%%writefile evaluation.py
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer


def retrieval_hit(docs, expected):
    if expected is None:
        return None
    return any(expected.lower() in d.page_content.lower() for d in docs)


def answer_similarity(ans, expected, embeddings):
    if expected is None:
        return None
    v1 = embeddings.embed_query(ans)
    v2 = embeddings.embed_query(expected)
    return cosine_similarity([v1], [v2])[0][0]


def rouge_score_calc(ans, expected):
    if expected is None:
        return None
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    return scorer.score(expected, ans)['rougeL'].fmeasure

Overwriting evaluation.py


In [18]:
%%writefile run_evaluation.py
import pandas as pd
from src.ingestion import ingest
from src.router import route_query
from src.retriever import retrieve
from src.generator import generate_answer
from src.evaluation import retrieval_hit, answer_similarity, rouge_score_calc
from test_questions import test_set


vectorstore, embeddings = ingest("data")

results = []

for sample in test_set:
    query = sample["query"]
    true_type = sample["type"]

    pred_type, reasoning = route_query(query, vectorstore, embeddings)

    if pred_type == "out_of_scope":
        answer = "OUT"
        docs = []
    else:
        docs = retrieve(query, vectorstore, pred_type)
        answer = generate_answer(query, docs, pred_type)

    results.append({
        "query": query,
        "true_type": true_type,
        "pred_type": pred_type,
        "routing_correct": pred_type == true_type,
        "retrieval": retrieval_hit(docs, sample["expected"]),
        "cosine": answer_similarity(answer, sample["expected"], embeddings),
        "rouge": rouge_score_calc(answer, sample["expected"])
    })

df = pd.DataFrame(results)
print(df)
df.to_csv("results/results.csv", index=False)

Overwriting run_evaluation.py


In [19]:
%%writefile requirements.txt
langchain
openai
faiss-cpu
scikit-learn
pandas
rouge-score

Overwriting requirements.txt
